# Domestic Flight Delay Records Analysis with PySpark
This notebook solves five tasks using the provided flight delay dataset and PySpark.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, hour, avg, max as spark_max
spark = SparkSession.builder.appName('FlightDelayAnalysis').getOrCreate()
df = spark.read.csv('Flight Dataset.csv', header=True, inferSchema=True)
df.show(5)

+--------+---------+---------+--------+--------+---------+---------+
| FL_DATE|DEP_DELAY|ARR_DELAY|AIR_TIME|DISTANCE| DEP_TIME| ARR_TIME|
+--------+---------+---------+--------+--------+---------+---------+
|1/1/2006|        5|       19|     350|    2475| 9.083333|12.483334|
|1/2/2006|      167|      216|     343|    2475|11.783334|15.766666|
|1/3/2006|       -7|       -2|     344|    2475| 8.883333|12.133333|
|1/4/2006|       -5|      -13|     331|    2475| 8.916667|    11.95|
|1/5/2006|       -3|      -17|     321|    2475|     8.95|11.883333|
+--------+---------+---------+--------+--------+---------+---------+
only showing top 5 rows


## Task 1: Number of flights that arrived earlier than expected

In [3]:
def count_early_arrivals(df):
    return df.filter(col('ARR_DELAY') < 0).count()

early_arrivals = count_early_arrivals(df)
print(f'Number of flights that arrived earlier than expected: {early_arrivals}')

Number of flights that arrived earlier than expected: 534655


## Task 2: Typical departure time for flights over 2000 miles

In [ ]:
def typical_departure_time_long_flights(df):
    long_flights = df.filter(col('DISTANCE') > 2000)
    long_flights = long_flights.withColumn('DEP_HOUR', (col('DEP_TIME')/100).cast('int'))
    return long_flights.groupBy('DEP_HOUR').count().orderBy(col('count').desc()).first()['DEP_HOUR']

typical_hour = typical_departure_time_long_flights(df)
print(f'Typical departure hour for flights over 2000 miles: {typical_hour}:00')

Typical departure hour for flights over 2000 miles: 0:00


## Task 3: Proportion of flights with arrival delays longer than 60 minutes

In [9]:
def proportion_long_arrival_delays(df):
    total = df.count()
    delayed = df.filter(col('ARR_DELAY') > 60).count()
    return delayed / total if total > 0 else 0
prop = proportion_long_arrival_delays(df)
print(f'Proportion of flights with arrival delays > 60 min: {prop:.2%}')

Proportion of flights with arrival delays > 60 min: 5.31%


## Task 4: Average airtime for flights that left earlier than 9:00 am

In [ ]:
def avg_airtime_early_departures(df):
    early = df.filter((col('DEP_TIME') < 900))
    return early.agg(avg(col('AIR_TIME'))).first()[0]
avg_airtime = avg_airtime_early_departures(df)
print(f'Average airtime for flights that left before 9:00 am: {avg_airtime:.2f} minutes')

Average airtime for flights that left before 9:00 am: 105.81 minutes


## Task 5: Maximum arrival delay for flights with no departure delay

In [ ]:
def max_arrival_delay_no_departure_delay(df):
    no_dep_delay = df.filter(col('DEP_DELAY') == 0)
    return no_dep_delay.agg(spark_max(col('ARR_DELAY'))).first()[0]
max_arrival_delay = max_arrival_delay_no_departure_delay(df)
print(f'Maximum arrival delay for flights with no departure delay: {max_arrival_delay} minutes')

Maximum arrival delay for flights with no departure delay: 232 minutes
